# 03_Fine_localization

In [ ]:
# Configuration -------------------------------------------------------------
import os, json, math, hashlib, itertools
import numpy as np, pandas as pd
import torch, torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

# Previous-notebook outputs (reused; never regenerated here).
RIPPLE_SETS_CSV  = run_path("HUMAID_ripple_sets.csv")
PROTECT_SETS_CSV = run_path("HUMAID_protect_sets.csv")
SUMMARY_CSV      = run_path("HUMAID_ripple_protect_summary.csv")
PREDICTION_CSV_CANDIDATES = [
    run_path("HUMAID_one_shot_domain_predictions.csv"),   # has exact_messages_json / exemplars_json
    run_path("HUMAID_all_predictions_per_instance.csv"),
]

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"    # localization only; weights never modified

# FiNE-faithful defaults (Appendix C).
TOP_K                 = 5    # neurons selected per source-object token (k = 5)
EXCLUDE_LAST_N_LAYERS = 3    # layer freezing: exclude last lf = 3 layers from localization
ONESHOT_PROMPT        = "domain"
ITERATION             = 0
WRONG_TOKEN           = "__wrong__"

# Outputs.
OUT_LOCALIZATION  = run_path("HUMAID_FiNE_ripple_localization.csv")
OUT_GROUP_SUMMARY = run_path("HUMAID_FiNE_group_neuron_summary.csv")
OUT_OVERLAP       = run_path("HUMAID_FiNE_group_overlap.csv")
OUT_LAYER_DIST    = run_path("HUMAID_FiNE_layer_distribution.csv")
OUT_FAILURES      = run_path("HUMAID_FiNE_localization_failures.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
print(f"device={DEVICE} dtype={DTYPE} | TOP_K={TOP_K} lf={EXCLUDE_LAST_N_LAYERS}")

# [OPT] Lightweight GPU-memory probe for the expensive stages (no-op on CPU).
def gpu_mem(tag=""):
    if torch.cuda.is_available():
        a = torch.cuda.memory_allocated()/1024**3
        r = torch.cuda.memory_reserved()/1024**3
        mx = torch.cuda.max_memory_allocated()/1024**3
        print(f"[gpu-mem] {tag:34s} alloc={a:6.2f}GiB reserved={r:6.2f}GiB max_alloc={mx:6.2f}GiB")


In [ ]:
# Load previous-notebook outputs (Ripple/Protect are only READ) -------------
assert os.path.exists(RIPPLE_SETS_CSV), f"missing {RIPPLE_SETS_CSV} (run previous notebook first)"
ripple_sets  = pd.read_csv(RIPPLE_SETS_CSV,  dtype=str, keep_default_na=False)
protect_sets = pd.read_csv(PROTECT_SETS_CSV, dtype=str, keep_default_na=False) \
               if os.path.exists(PROTECT_SETS_CSV) else pd.DataFrame(columns=["sample_id"])
summary_prev = pd.read_csv(SUMMARY_CSV) if os.path.exists(SUMMARY_CSV) else None
assert len(ripple_sets) > 0, "ripple_sets.csv is empty; nothing to localize."

# Prediction CSV supplies the EXACT original One-shot/domain prompt metadata.
pred_path = next((p for p in PREDICTION_CSV_CANDIDATES if os.path.exists(p)), None)
assert pred_path is not None, "No predictions CSV found for exact-prompt reconstruction."
pred = pd.read_csv(pred_path, dtype=str, keep_default_na=False)
print(f"[load] ripple_sets={len(ripple_sets)} rows | protect_sets={len(protect_sets)} rows "
      f"| predictions={pred_path} ({len(pred)} rows)")

# Restrict prediction metadata to the SAME controlled scope.
scope = pred.copy()
for col, val in [("shot", "one"), ("prompt", ONESHOT_PROMPT), ("iteration", str(ITERATION))]:
    if col in scope.columns:
        scope = scope[scope[col].astype(str) == val]
meta_cols = [c for c in ["sample_id", "exact_messages_json", "exemplars_json",
                         "raw_model_output", "true_label", "predicted_label"] if c in scope.columns]
pred_meta = scope[meta_cols].drop_duplicates("sample_id").set_index("sample_id")
print(f"[load] prompt metadata available for {len(pred_meta)} samples in scope "
      f"(cols: {list(pred_meta.columns)})")

RIPPLE_IDS  = set(ripple_sets["sample_id"].astype(str))
PROTECT_IDS = set(protect_sets["sample_id"].astype(str)) if len(protect_sets) else set()
print(f"[load] {ripple_sets['ripple_group_id'].nunique()} ripple groups | "
      f"{len(RIPPLE_IDS)} ripple samples | {len(PROTECT_IDS)} protect samples")


In [ ]:
# Load Qwen2.5-3B-Instruct and inspect the real MLP architecture ------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE)
model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad_(False)

cfg     = model.config
HIDDEN  = cfg.hidden_size            # d_h
INTER   = cfg.intermediate_size      # d_m
NLAYERS = cfg.num_hidden_layers      # L
VOCAB   = cfg.vocab_size             # v
print(f"config: hidden_act={cfg.hidden_act} d_h={HIDDEN} d_m={INTER} L={NLAYERS} v={VOCAB}")
assert str(cfg.hidden_act).lower() in ("silu", "swish"), f"unexpected act {cfg.hidden_act}"

layers = model.model.layers
mlp0 = layers[0].mlp
for name in ("gate_proj", "up_proj", "down_proj"):
    assert hasattr(mlp0, name), f"Qwen MLP missing {name}"

# FiNE W^l_out == down_proj.weight (d_h x d_m); q^l == input to down_proj.
W_out0 = layers[0].mlp.down_proj.weight
assert tuple(W_out0.shape) == (HIDDEN, INTER), f"down_proj.weight {tuple(W_out0.shape)} != (d_h,d_m)"
W_u = model.get_output_embeddings().weight          # unembedding, v x d_h
assert tuple(W_u.shape) == (VOCAB, HIDDEN), f"W_u {tuple(W_u.shape)} != (v,d_h)"

d_h, d_m = HIDDEN, INTER
allowed_layers = list(range(0, NLAYERS - EXCLUDE_LAST_N_LAYERS))   # first L - lf layers
assert allowed_layers, "EXCLUDE_LAST_N_LAYERS too large"
print(f"W^l_out = down_proj.weight : ({HIDDEN}, {INTER}) = (d_h, d_m)  [asserted]")
print(f"W_u     = output embedding : ({VOCAB}, {HIDDEN}) = (v, d_h)   [asserted]")
print(f"allowed layers (localization): {allowed_layers[0]}..{allowed_layers[-1]} "
      f"({len(allowed_layers)} of {NLAYERS}; last {EXCLUDE_LAST_N_LAYERS} frozen)")


In [ ]:
# Snapshot model weights (to prove they are unchanged at the end) -----------
@torch.no_grad()
def weight_digest(model):
    """Order-independent-per-param SHA256 over (name, sum, sumsq, min, max) in float64."""
    h = hashlib.sha256()
    for n, p in model.named_parameters():
        d = p.detach()
        stat = torch.stack([d.double().sum(), (d.double()**2).sum(),
                            d.double().amin(), d.double().amax()]).cpu().numpy()
        h.update(n.encode("utf-8")); h.update(stat.tobytes())
    return h.hexdigest()

DIGEST_BEFORE = weight_digest(model)
print("weight digest BEFORE localization:", DIGEST_BEFORE)


In [ ]:
# down_proj-input capture + exact-prompt reconstruction ---------------------
# Forward-PRE-hook on down_proj captures its input == q^l = silu(gate_proj)*up_proj.
_capture = {}
def _make_pre_hook(l):
    def hook(module, args):
        _capture[l] = args[0][:, -1, :].detach().to(torch.float32)   # [1, d_m], last token
    return hook
_hook_handles = [layers[l].mlp.down_proj.register_forward_pre_hook(_make_pre_hook(l))
                 for l in allowed_layers]
print(f"registered {len(_hook_handles)} down_proj pre-hooks (allowed layers only)")

def _norm(s):  # whitespace/case-insensitive substring matching
    return " ".join(str(s).split()).casefold()

def reconstruct_prompt(exact_messages_json):
    """Return (prompt_str, input_ids) for the EXACT One-shot/domain generation context.
    Both the activation forward pass AND the contextual source tokenization are derived from
    the SAME prompt_str, so the prompt tokenization is internally consistent."""
    obj = json.loads(exact_messages_json)
    if isinstance(obj, list):                       # list of {role, content} -> chat template
        prompt_str = tokenizer.apply_chat_template(obj, tokenize=False, add_generation_prompt=True)
    else:                                           # already-formatted string fallback
        prompt_str = str(obj)
    input_ids = tokenizer(prompt_str, add_special_tokens=False, return_tensors="pt").input_ids
    return prompt_str, input_ids

def contextual_source_token_ids(prompt_str, label_text):
    """FiNE source-object token ids computed IN CONTEXT: tokenize (prompt_str + label_text),
    then strip the prompt prefix and return the label's suffix ids. Never tokenizes the label
    in isolation. Returns (source_ids, prefix_clean)."""
    prompt_only = tokenizer(prompt_str, add_special_tokens=False)["input_ids"]
    full        = tokenizer(prompt_str + label_text, add_special_tokens=False)["input_ids"]
    cpl = 0
    for a, b in zip(prompt_only, full):
        if a == b: cpl += 1
        else: break
    source_ids   = full[cpl:]
    prefix_clean = (cpl == len(prompt_only))   # False => boundary merge (rare); flagged, not silent
    return source_ids, prefix_clean

@torch.no_grad()
def last_token_activations(input_ids):
    _capture.clear()
    model(input_ids=input_ids.to(DEVICE))
    return {l: _capture[l][0] for l in allowed_layers}    # each [d_m] on DEVICE, fp32


In [ ]:
# FiNE Eq. (3): per-token neuron contributions + global top-k ---------------
@torch.no_grad()
def topk_neurons_for_token(q_by_layer, token_id):
    """Return top-TOP_K (layer_id, neuron_id, score) ranked GLOBALLY across allowed layers."""
    wu = W_u[token_id].to(torch.float32)                       # [d_h]  = W_u[t,:]
    per_layer = []
    for l in allowed_layers:
        W_out = layers[l].mlp.down_proj.weight.to(torch.float32)   # [d_h, d_m] = W^l_out
        proj  = wu @ W_out                                          # [d_m] = (W_u W^l_out)_{t,:}
        per_layer.append(q_by_layer[l] * proj)                     # Eq.(3), elementwise over neurons
    C = torch.stack(per_layer, 0)                                  # [num_allowed, d_m]
    k = min(TOP_K, C.numel())
    vals, idx = torch.topk(C.reshape(-1), k)                       # GLOBAL ranking (not per-layer)
    layer_pos = (idx // d_m).tolist(); neuron_ids = (idx % d_m).tolist()
    return [(allowed_layers[p], int(n), float(v))
            for p, n, v in zip(layer_pos, neuron_ids, vals.tolist())]

@torch.no_grad()
def topk_neurons_for_tokens(q_by_layer, token_ids):
    """[OPT] Batched, BIT-IDENTICAL equivalent of calling topk_neurons_for_token per token.
    Converts each allowed layer's down_proj.weight to fp32 ONCE per sample (not once per token)
    and reuses it for every source token. Each per-token score uses the SAME vector-matrix product
    (W_u[t].float() @ W_out.float()) and the SAME global top-k as before, so selected neurons are
    identical. Returns a list (per token) of [(layer_id, neuron_id, score), ...]."""
    token_ids = list(token_ids)
    T = len(token_ids)
    WU = [W_u[t].to(torch.float32) for t in token_ids]                 # each [d_h]
    Cs = [torch.empty(len(allowed_layers), d_m, dtype=torch.float32, device=WU[0].device)
          for _ in range(T)]
    for li, l in enumerate(allowed_layers):
        W_out = layers[l].mlp.down_proj.weight.to(torch.float32)       # convert ONCE per sample/layer
        ql = q_by_layer[l]
        for ti in range(T):
            proj = WU[ti] @ W_out                                      # [d_m] == original wu @ W_out
            Cs[ti][li] = ql * proj                                     # Eq.(3), same as original
        del W_out
    out = []
    for ti in range(T):
        C = Cs[ti]                                                     # [num_allowed, d_m]
        k = min(TOP_K, C.numel())
        vals, idx = torch.topk(C.reshape(-1), k)                       # GLOBAL ranking (identical)
        layer_pos = (idx // d_m).tolist(); neuron_ids = (idx % d_m).tolist()
        out.append([(allowed_layers[p], int(n), float(v))
                    for p, n, v in zip(layer_pos, neuron_ids, vals.tolist())])
    return out

@torch.no_grad()
def verify_q_mapping(input_ids):
    """Assert the captured down_proj input equals silu(gate_proj)*up_proj (gated-MLP q^l)."""
    l0 = allowed_layers[0]; buf = {}
    h1 = layers[l0].mlp.gate_proj.register_forward_hook(lambda m,a,o: buf.__setitem__("g", o[:, -1, :].float()))
    h2 = layers[l0].mlp.up_proj.register_forward_hook(  lambda m,a,o: buf.__setitem__("u", o[:, -1, :].float()))
    _capture.clear(); model(input_ids=input_ids.to(DEVICE)); h1.remove(); h2.remove()
    ok = torch.allclose(F.silu(buf["g"]) * buf["u"], _capture[l0], atol=1e-2, rtol=1e-2)
    print(f"[faithfulness] layer {l0}: down_proj input == silu(gate_proj)*up_proj -> {bool(ok)}")
    assert ok, "q^l mapping check FAILED (down_proj input != silu(gate)*up)"


In [ ]:
# Localize EACH Ripple member independently (Algorithm 1) -------------------
localization_rows, failure_rows, verify_report = [], [], []
if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
gpu_mem("FiNE localize: start")
source_recon = {}          # sample_id -> decoded contextual source-object string (sanity #3)
_faithfulness_done = False

def _fail(gid, sid, reason, **extra):
    failure_rows.append({"ripple_group_id": gid, "sample_id": sid, "reason": reason, **extra})

for _, r in ripple_sets.iterrows():
    sid  = str(r["sample_id"]); gid = str(r["ripple_group_id"])
    true_label = str(r["true_label"]); pred_label = str(r["predicted_label"]); text = str(r["text"])

    if pred_label == WRONG_TOKEN or pred_label == true_label:
        _fail(gid, sid, "not_a_valid_misclassification", predicted_label=pred_label); continue
    if sid not in pred_meta.index or "exact_messages_json" not in pred_meta.columns \
       or not str(pred_meta.loc[sid, "exact_messages_json"]):
        _fail(gid, sid, "missing_exact_messages"); continue

    try:
        prompt_str, input_ids = reconstruct_prompt(pred_meta.loc[sid, "exact_messages_json"])
    except Exception as e:
        _fail(gid, sid, f"reconstruction_error:{type(e).__name__}"); continue

    # Verify reconstructed context corresponds to THIS sample (tweet text must be present).
    if _norm(text) not in _norm(prompt_str):
        _fail(gid, sid, "prompt_verification_failed"); continue
    # One-shot exemplar presence (non-fatal, recorded).
    exemplar_ok = None
    if "exemplars_json" in pred_meta.columns and pred_meta.loc[sid, "exemplars_json"]:
        try:
            exs = json.loads(pred_meta.loc[sid, "exemplars_json"])
            ex_texts = []
            if isinstance(exs, list):
                for e in exs:
                    if isinstance(e, dict):
                        ex_texts += [str(v) for k, v in e.items() if "text" in k.lower()]
                    else:
                        ex_texts.append(str(e))
            exemplar_ok = (any(_norm(t) in _norm(prompt_str) for t in ex_texts)
                           if ex_texts else None)
        except Exception:
            exemplar_ok = None

    # Source object o = current WRONG predicted_label, tokenized IN CONTEXT (Algorithm 1).
    standalone_ids = tokenizer(pred_label, add_special_tokens=False)["input_ids"]   # display only
    src_ids, prefix_clean = contextual_source_token_ids(prompt_str, pred_label)
    if len(src_ids) == 0:
        _fail(gid, sid, "empty_source_tokenization"); continue
    decoded_ctx = tokenizer.decode(src_ids)
    if _norm(decoded_ctx) != _norm(pred_label):
        _fail(gid, sid, "contextual_tokenization_mismatch",
              decoded=decoded_ctx, source_ids=str(src_ids)); continue
    source_recon[sid] = decoded_ctx

    # Required per-sample verification print.
    print(f"[src-tok] sample {sid} | predicted_label={pred_label!r}")
    print(f"          standalone  ids = {standalone_ids}")
    print(f"          contextual  ids = {src_ids}  (prefix_clean={prefix_clean})")
    print(f"          decoded ctx     = {decoded_ctx!r}")

    q_by_layer = last_token_activations(input_ids)
    if not _faithfulness_done:
        verify_q_mapping(input_ids); _faithfulness_done = True

    _tok_topk = topk_neurons_for_tokens(q_by_layer, src_ids)   # [OPT] batched; bit-identical to per-token
    for t_id, _tk in zip(src_ids, _tok_topk):
        tok_str = tokenizer.decode([t_id])
        for rank, (lid, nid, score) in enumerate(_tk, start=1):
            if not math.isfinite(score):
                _fail(gid, sid, "non_finite_score", source_token_id=int(t_id)); continue
            localization_rows.append({
                "ripple_group_id": gid, "sample_id": sid, "text": text,
                "true_label": true_label, "predicted_label": pred_label,
                "source_token": tok_str, "source_token_id": int(t_id),
                "layer_id": int(lid), "neuron_id": int(nid),
                "contribution_score": float(score), "rank": int(rank),
            })
    verify_report.append({"sample_id": sid, "tweet_found": True, "exemplar_found": exemplar_ok,
                          "n_source_tokens": len(src_ids), "prefix_clean": bool(prefix_clean)})

fine_local = pd.DataFrame(localization_rows)
failures   = pd.DataFrame(failure_rows)
gpu_mem("FiNE localize: end")
print(f"\nlocalized: {len(fine_local)} rows over "
      f"{fine_local['sample_id'].nunique() if len(fine_local) else 0} samples / "
      f"{fine_local['ripple_group_id'].nunique() if len(fine_local) else 0} groups | "
      f"failures: {len(failures)}")
if len(verify_report):
    vr = pd.DataFrame(verify_report)
    print(f"prompt verification: tweet_found={int(vr['tweet_found'].sum())}/{len(vr)} | "
          f"exemplar_found(True)={int((vr['exemplar_found']==True).sum())} | "
          f"prefix_clean(all)={bool(vr['prefix_clean'].all())}")


In [ ]:
# Save per-neuron localization table ----------------------------------------
loc_cols = ["ripple_group_id","sample_id","text","true_label","predicted_label",
            "source_token","source_token_id","layer_id","neuron_id","contribution_score","rank"]
(fine_local.reindex(columns=loc_cols) if len(fine_local)
 else pd.DataFrame(columns=loc_cols)).to_csv(OUT_LOCALIZATION, index=False)
print(f"saved {OUT_LOCALIZATION} ({len(fine_local)} rows)")


In [ ]:
# Group-level neuron analysis (ANALYSIS ONLY; nothing is edited) ------------
# Per-member neuron SET = union of (layer_id, neuron_id) across that member's source tokens.
member_neurons = {}
for (gid, sid), g in fine_local.groupby(["ripple_group_id", "sample_id"]):
    member_neurons[(gid, sid)] = set(map(tuple, g[["layer_id", "neuron_id"]].values.tolist()))
group_size = {gid: g["sample_id"].nunique() for gid, g in fine_local.groupby("ripple_group_id")}

summary_rows = []
for gid, g in fine_local.groupby("ripple_group_id"):
    gsize = group_size[gid]
    for (lid, nid), gg in g.groupby(["layer_id", "neuron_id"]):
        mem = gg["sample_id"].nunique()
        summary_rows.append({
            "ripple_group_id": gid, "layer_id": int(lid), "neuron_id": int(nid),
            "member_count": int(mem), "member_fraction": mem / gsize,
            "mean_contribution": float(gg["contribution_score"].mean()),
            "max_contribution":  float(gg["contribution_score"].max()),
        })
group_summary = (pd.DataFrame(summary_rows)
                 .sort_values(["ripple_group_id", "member_count", "max_contribution"],
                              ascending=[True, False, False]).reset_index(drop=True)) \
                if summary_rows else pd.DataFrame(
                 columns=["ripple_group_id","layer_id","neuron_id","member_count",
                          "member_fraction","mean_contribution","max_contribution"])
group_summary.to_csv(OUT_GROUP_SUMMARY, index=False)
print(f"saved {OUT_GROUP_SUMMARY} ({len(group_summary)} neuron rows)")


In [ ]:
# --- 9. Pairwise Jaccard overlap between members' localized neuron sets -----------
overlap_rows = []
for gid, g in fine_local.groupby("ripple_group_id"):
    sids = sorted(g["sample_id"].unique())
    for a, b in itertools.combinations(sids, 2):
        Sa, Sb = member_neurons[(gid, a)], member_neurons[(gid, b)]
        inter, uni = len(Sa & Sb), len(Sa | Sb)
        overlap_rows.append({
            "ripple_group_id": gid, "sample_id_a": a, "sample_id_b": b,
            "intersection_size": int(inter), "union_size": int(uni),
            "jaccard": (inter / uni if uni else float("nan")),
        })
group_overlap = pd.DataFrame(overlap_rows) if overlap_rows else pd.DataFrame(
    columns=["ripple_group_id","sample_id_a","sample_id_b","intersection_size","union_size","jaccard"])
group_overlap.to_csv(OUT_OVERLAP, index=False)
print(f"saved {OUT_OVERLAP} ({len(group_overlap)} member pairs)")


In [ ]:
# Layer distribution of localized neurons per group ------------------------
layer_rows = []
for gid, g in fine_local.groupby("ripple_group_id"):
    gsize = group_size[gid]
    for lid, gl in g.groupby("layer_id"):
        layer_rows.append({
            "ripple_group_id": gid, "layer_id": int(lid),
            "selected_neuron_count": int(gl[["neuron_id"]].drop_duplicates().shape[0]),
            # distinct Ripple members that have >=1 selected neuron IN THIS LAYER (not group size)
            "member_count": int(gl["sample_id"].nunique()),
        })
layer_dist = (pd.DataFrame(layer_rows).sort_values(["ripple_group_id","layer_id"]).reset_index(drop=True)
              if layer_rows else pd.DataFrame(
              columns=["ripple_group_id","layer_id","selected_neuron_count","member_count"]))
layer_dist.to_csv(OUT_LAYER_DIST, index=False)

# Failures table.
fail_cols = ["ripple_group_id","sample_id","reason"]
failures_out = failures.reindex(columns=fail_cols + [c for c in failures.columns if c not in fail_cols]) \
               if len(failures) else pd.DataFrame(columns=fail_cols)
failures_out.to_csv(OUT_FAILURES, index=False)
print(f"saved {OUT_LAYER_DIST} ({len(layer_dist)} rows) | {OUT_FAILURES} ({len(failures_out)} rows)")


In [ ]:
# Verify weights unchanged + sanity checks ---------------------------------
DIGEST_AFTER = weight_digest(model)
print("weight digest AFTER localization:", DIGEST_AFTER)
assert DIGEST_AFTER == DIGEST_BEFORE, "MODEL WEIGHTS CHANGED DURING LOCALIZATION!"
print("[sanity 9] model weights identical before/after: OK")

if len(fine_local):
    loc_ids = set(fine_local["sample_id"].astype(str))
    # 1 every localized sample is a Ripple sample
    assert loc_ids <= RIPPLE_IDS, "localized a non-Ripple sample"
    # 2 every localized sample is misclassified
    assert (fine_local["predicted_label"] != fine_local["true_label"]).all(), "correct sample localized"
    # 3 the localized source object IS the current predicted_label
    for sid in loc_ids:
        pl = fine_local.loc[fine_local["sample_id"] == sid, "predicted_label"].iloc[0]
        assert _norm(source_recon[sid]) == _norm(pl), f"source object != predicted_label for {sid}"
    # 4 no Protect sample was localized
    assert loc_ids.isdisjoint(PROTECT_IDS), "a Protect sample was localized"
    # 5 valid layer ids (within allowed, last lf excluded)
    assert fine_local["layer_id"].between(0, NLAYERS - EXCLUDE_LAST_N_LAYERS - 1).all()
    assert set(fine_local["layer_id"]).issubset(set(allowed_layers))
    # 6 valid neuron ids
    assert fine_local["neuron_id"].between(0, d_m - 1).all()
    # 7 finite scores
    assert np.isfinite(fine_local["contribution_score"].to_numpy()).all()
    # 8 ripple group ids preserved + group->true_label mapping consistent
    assert set(fine_local["ripple_group_id"]).issubset(set(ripple_sets["ripple_group_id"].astype(str)))
    prev_map = ripple_sets.groupby("ripple_group_id")["true_label"].agg(lambda s: set(s))
    for gid, g in fine_local.groupby("ripple_group_id"):
        assert set(g["true_label"]) <= prev_map.loc[gid], f"group {gid} true_label drift"
    print(f"[sanity 1-8] all localization invariants hold "
          f"({len(loc_ids)} samples, {len(fine_local)} neuron rows).")
else:
    print("[sanity] no localized rows (check HUMAID_FiNE_localization_failures.csv).")


In [ ]:
# Human-readable localization + group overlap summary ----------------------
def _short(t, n=140):
    t = str(t).replace("\n", " ");  return t if len(t) <= n else t[:n] + "..."

for gid in sorted(fine_local["ripple_group_id"].unique(), key=lambda x: (len(x), x)) if len(fine_local) else []:
    g = fine_local[fine_local["ripple_group_id"] == gid]
    print("#"*90)
    print(f"RIPPLE GROUP: {gid}   (true_label = {g['true_label'].iloc[0]}, "
          f"members = {g['sample_id'].nunique()})")
    for sid in sorted(g["sample_id"].unique()):
        gs = g[g["sample_id"] == sid]
        print("-"*90)
        print(f"Sample ID: {sid}")
        print(f"  Text: {_short(gs['text'].iloc[0])}")
        print(f"  True label            : {gs['true_label'].iloc[0]}")
        print(f"  Current wrong prediction: {gs['predicted_label'].iloc[0]}")
        for tok, gt in gs.groupby("source_token", sort=False):
            print(f"  Source token: {tok!r}  (id={int(gt['source_token_id'].iloc[0])})")
            for _, row in gt.sort_values("rank").iterrows():
                print(f"    Layer {int(row['layer_id']):>2} | Neuron {int(row['neuron_id']):>5} "
                      f"| Contribution {row['contribution_score']:.4f}")
    # group-level overlap
    print("."*90)
    sids = sorted(g["sample_id"].unique())
    sets = {s: member_neurons[(gid, s)] for s in sids}
    inter_all = set.intersection(*sets.values()) if len(sets) > 1 else set()
    from collections import Counter
    freq = Counter()
    for s in sids: freq.update(sets[s])
    shared = {k: c for k, c in freq.items() if c >= 2}
    atleast50 = {k: c for k, c in freq.items() if c / len(sids) >= 0.5}
    print(f"  neurons shared by >=2 members : {len(shared)}")
    print(f"  neurons in >=50% of members   : {len(atleast50)}")
    print(f"  intersection across ALL members: {sorted(inter_all)[:10]}"
          + (" ..." if len(inter_all) > 10 else ""))
    ov = group_overlap[group_overlap["ripple_group_id"] == gid]
    if len(ov):
        print(f"  pairwise Jaccard: mean={ov['jaccard'].mean():.3f} "
              f"min={ov['jaccard'].min():.3f} max={ov['jaccard'].max():.3f}")
    ld = layer_dist[layer_dist["ripple_group_id"] == gid].sort_values("selected_neuron_count", ascending=False)
    top_layers = ", ".join(f"L{int(r.layer_id)}:{int(r.selected_neuron_count)}" for _, r in ld.head(6).iterrows())
    print(f"  layer distribution (top): {top_layers}")
print("\nLocalization analysis complete.")
